In [13]:
# ------------------------------------------------------------------------------
# 1. IMPORT THƯ VIỆN
# ------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from scipy.special import boxcox1p
from pathlib import Path

# Import các module từ Scikit-learn
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor, StackingRegressor
from sklearn.model_selection import KFold, cross_val_score

# Import các thư viện Boosting
import xgboost as xgb
import lightgbm as lgb

# ------------------------------------------------------------------------------
# 2. THIẾT LẬP ĐƯỜNG DẪN & LOAD DỮ LIỆU
# ------------------------------------------------------------------------------
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir

DATA_DIR = project_root / 'data'
SUBMISSIONS_DIR = project_root / 'submissions'
TRAIN_FILE = DATA_DIR / 'train.csv'
TEST_FILE = DATA_DIR / 'test.csv'

print(f"Data directory: {DATA_DIR}")
print(f"Train file exists: {TRAIN_FILE.exists()}")

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

# ------------------------------------------------------------------------------
# 3. XỬ LÝ DỮ LIỆU & FEATURE ENGINEERING
# ------------------------------------------------------------------------------

# --- Xử lý Outliers ---
train_df = train_df.drop(train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300000)].index)

# --- Xử lý Target (SalePrice) ---
train_df["SalePrice"] = np.log1p(train_df["SalePrice"])
y_train = train_df.SalePrice.values

# Lưu ID và bỏ cột Id
train_ID = train_df['Id']
test_ID = test_df['Id']
train_df.drop("Id", axis=1, inplace=True)
test_df.drop("Id", axis=1, inplace=True)

# Gộp train và test
ntrain = train_df.shape[0]
ntest = test_df.shape[0]
all_data = pd.concat((train_df, test_df)).reset_index(drop=True)
all_data.drop(['SalePrice'], axis=1, inplace=True)

# --- Điền giá trị thiếu (Missing Values) ---
cols_none = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", 
             "GarageType", "GarageFinish", "GarageQual", "GarageCond",
             "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
             "MasVnrType", "MSSubClass"]
for col in cols_none:
    all_data[col] = all_data[col].fillna("None")

cols_zero = ["GarageYrBlt", "GarageArea", "GarageCars",
             "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF", 
             "MasVnrArea", "BsmtFullBath", "BsmtHalfBath"]
for col in cols_zero:
    all_data[col] = all_data[col].fillna(0)

all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median()))

cols_mode = ["MSZoning", "Electrical", "KitchenQual", "Exterior1st", "Exterior2nd", "SaleType"]
for col in cols_mode:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

all_data["Functional"] = all_data["Functional"].fillna("Typ")
all_data = all_data.drop(['Utilities'], axis=1)

# --- Tạo Feature Mới ---
all_data['MSSubClass'] = all_data['MSSubClass'].apply(str)
all_data['OverallCond'] = all_data['OverallCond'].astype(str)
all_data['YrSold'] = all_data['YrSold'].astype(str)
all_data['MoSold'] = all_data['MoSold'].astype(str)

all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# Label Encoding
cols_ordinal = ('FireplaceQu', 'BsmtQual', 'BsmtCond', 'GarageQual', 'GarageCond', 
                'ExterQual', 'ExterCond','HeatingQC', 'PoolQC', 'KitchenQual', 
                'BsmtFinType1', 'BsmtFinType2', 'Functional', 'Fence', 'BsmtExposure', 
                'GarageFinish', 'LandSlope', 'LotShape', 'PavedDrive', 'Street', 
                'Alley', 'CentralAir', 'MSSubClass', 'OverallCond', 'YrSold', 'MoSold')

for c in cols_ordinal:
    lbl = LabelEncoder() 
    lbl.fit(list(all_data[c].values)) 
    all_data[c] = lbl.transform(list(all_data[c].values))

# --- Xử lý Skewness ---
numeric_feats = all_data.dtypes[all_data.dtypes != "object"].index
skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewness = pd.DataFrame({'Skew' :skewed_feats})
skewness = skewness[abs(skewness) > 0.75]

skewed_features = skewness.index
lam = 0.15
for feat in skewed_features:
    all_data[feat] = boxcox1p(all_data[feat], lam)

all_data = pd.get_dummies(all_data)
X_train = all_data[:ntrain]
X_test = all_data[ntrain:]

print(f"Dữ liệu sẵn sàng. Train: {X_train.shape}, Test: {X_test.shape}")

# ------------------------------------------------------------------------------
# 4. THIẾT LẬP MÔ HÌNH
# ------------------------------------------------------------------------------
lasso = make_pipeline(RobustScaler(), Lasso(alpha =0.0005, random_state=1))
ENet = make_pipeline(RobustScaler(), ElasticNet(alpha=0.0005, l1_ratio=.9, random_state=3))
GBoost = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05,
                                   max_depth=4, max_features='sqrt',
                                   min_samples_leaf=15, min_samples_split=10, 
                                   loss='huber', random_state =5)
lgb_model = lgb.LGBMRegressor(objective='regression',num_leaves=5,
                              learning_rate=0.05, n_estimators=720,
                              max_bin = 55, bagging_fraction = 0.8,
                              bagging_freq = 5, feature_fraction = 0.2319,
                              feature_fraction_seed=9, bagging_seed=9,
                              min_data_in_leaf =6, min_sum_hessian_in_leaf = 11, verbose=-1)

xgb_model = xgb.XGBRegressor(colsample_bytree=0.4603, gamma=0.0468, 
                             learning_rate=0.05, max_depth=3, 
                             min_child_weight=1.7817, n_estimators=2200,
                             reg_alpha=0.4640, reg_lambda=0.8571,
                             subsample=0.5213, silent=1,
                             random_state =7, nthread = -1)

# Stacking Model
estimators = [('lasso', lasso), ('enet', ENet), ('gb', GBoost), ('lgb', lgb_model)]
stack_model = StackingRegressor(estimators=estimators, final_estimator=xgb_model)

# ------------------------------------------------------------------------------
# 5. TÍNH TOÁN VÀ IN CHỈ SỐ RMSE (ĐÃ THÊM MỚI)
# ------------------------------------------------------------------------------
print("\n=== Đang tính toán chỉ số RMSE (Cross-Validation 5-folds) ===")
print("Quá trình này có thể mất vài phút, vui lòng đợi...")

# Định nghĩa hàm Cross-Validation
n_folds = 5
kf = KFold(n_folds, shuffle=True, random_state=42)

def rmsle_cv(model):
    # scoring='neg_mean_squared_error' trả về số âm, nên cần thêm dấu trừ
    rmse = np.sqrt(-cross_val_score(model, X_train.values, y_train, scoring="neg_mean_squared_error", cv=kf))
    return(rmse)

# Tính và in kết quả
score = rmsle_cv(stack_model)
print(f"\nResult: Stacking RMSE score: {score.mean():.4f} (std: {score.std():.4f})\n")

# ------------------------------------------------------------------------------
# 6. HUẤN LUYỆN VÀ DỰ ĐOÁN CUỐI CÙNG
# ------------------------------------------------------------------------------
print("=== Đang huấn luyện lại mô hình trên toàn bộ dữ liệu ===")
stack_model.fit(X_train, y_train)

print("Đang dự đoán...")
final_predictions = np.expm1(stack_model.predict(X_test))

submission = pd.DataFrame()
submission['Id'] = test_ID
submission['SalePrice'] = final_predictions

if not SUBMISSIONS_DIR.exists():
    SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
submission_path = SUBMISSIONS_DIR / 'cat.csv'
submission.to_csv(submission_path, index=False)
print(f"Hoàn tất! Đã lưu file kết quả tại: {submission_path}")

Data directory: D:\Library\PTDLKD\house_price_analysis\house_price_analysis\data
Train file exists: True
Dữ liệu sẵn sàng. Train: (1458, 220), Test: (1459, 220)

=== Đang tính toán chỉ số RMSE (Cross-Validation 5-folds) ===
Quá trình này có thể mất vài phút, vui lòng đợi...


D:\Library\PTDLKD\house_price_analysis\house_price_analysis\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Library\PTDLKD\house_price_analysis\house_price_analysis\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Library\PTDLKD\house_price_analysis\house_price_analysis\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Library\PTDLKD\house_price_analysis\house_price_analysis\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
D:\Library\PTDLKD\house_price_analysis\house_price_analysis\.venv\li


Result: Stacking RMSE score: 0.1101 (std: 0.0106)

=== Đang huấn luyện lại mô hình trên toàn bộ dữ liệu ===


D:\Library\PTDLKD\house_price_analysis\house_price_analysis\.venv\lib\site-packages\xgboost\training.py:199: UserWarning: [21:49:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Đang dự đoán...
Hoàn tất! Đã lưu file kết quả tại: D:\Library\PTDLKD\house_price_analysis\house_price_analysis\submissions\cat.csv
